# Visualização da Informação — Projeto da Disciplina

## Dataset: Taxa de Câmbio USD/BRL (2010–2019)

**Fonte:** [Investing.com — USD/BRL Historical Data](https://br.investing.com/currencies/usd-brl-historical-data)

O dataset contém cotações diárias do dólar americano (USD) frente ao real brasileiro (BRL) ao longo de 10 anos (jan/2010 a dez/2019), totalizando aproximadamente 2.608 registros.

### Técnicas utilizadas (uma de cada unidade):
| Unidade | Técnica | Gráfico |
|---|---|---|
| Visualização de Informação Temporal | Gráfico de Linhas | Evolução diária USD/BRL (2010–2019) |
| Visualização com gráficos da Estatística Descritiva | Gráfico de Barras | Média anual e dispersão (±desvio padrão) |
| Visualização de informação hierárquica / relacional | Heatmap (Mapa de Calor) | Variação mensal média por ano |


In [ ]:
# ─────────────────────────────────────────────
#  Importações e configuração global
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# Estilo geral
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.labelsize':   11,
})

ACCENT   = '#58a6ff'   # azul destaque
ACCENT2  = '#f78166'   # laranja/vermelho
ACCENT3  = '#3fb950'   # verde
MUTED    = '#8b949e'   # cinza suave

print('Bibliotecas carregadas com sucesso ✓')

In [ ]:
# ─────────────────────────────────────────────
#  Carregamento e preparação dos dados
# ─────────────────────────────────────────────
df = pd.read_csv('USD_BRL_hist.csv', sep=',', quotechar='"')
df.columns = ['Data', 'USD_BRL']
df['Data']    = pd.to_datetime(df['Data'], format='%d.%m.%Y')
df            = df.dropna().sort_values('Data').reset_index(drop=True)

# Colunas derivadas
df['Ano']     = df['Data'].dt.year
df['Mes']     = df['Data'].dt.month
df['NomeMes'] = df['Data'].dt.strftime('%b')   # Jan, Feb …

print(f'Registros: {len(df):,}')
print(f'Período:   {df["Data"].min().date()} → {df["Data"].max().date()}')
print(f'Min BRL:   R$ {df["USD_BRL"].min():.4f}  |  Max BRL: R$ {df["USD_BRL"].max():.4f}')
df.tail()

---
## Gráfico 1 — Visualização Temporal: Gráfico de Linhas

**Justificativa:** dados de séries temporais (cotações diárias ao longo de 10 anos) são naturalmente representados por gráfico de linhas, que evidencia tendências, sazonalidades e eventos pontuais ao longo do tempo.

In [ ]:
# ─────────────────────────────────────────────
#  Gráfico 1 — Série temporal diária USD/BRL
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Média móvel de 30 dias
mm30 = df['USD_BRL'].rolling(window=30).mean()

# Linha principal
ax.plot(df['Data'], df['USD_BRL'],
        color=ACCENT, linewidth=0.8, alpha=0.55, label='Cotação diária')

# Média móvel
ax.plot(df['Data'], mm30,
        color=ACCENT2, linewidth=2.0, label='Média móvel 30 dias')

# Área de preenchimento
ax.fill_between(df['Data'], df['USD_BRL'], df['USD_BRL'].min(),
                alpha=0.08, color=ACCENT)

# Anotação do pico
idx_max = df['USD_BRL'].idxmax()
ax.annotate(
    f'Máximo\nR$ {df.loc[idx_max,"USD_BRL"]:.2f} ({df.loc[idx_max,"Data"].strftime("%b/%Y")})',
    xy=(df.loc[idx_max,'Data'], df.loc[idx_max,'USD_BRL']),
    xytext=(pd.Timestamp('2016-06-01'), 4.6),
    fontsize=8.5, color=ACCENT2,
    arrowprops=dict(arrowstyle='->', color=ACCENT2, lw=1.2),
    bbox=dict(boxstyle='round,pad=0.3', fc='#161b22', ec=ACCENT2, lw=0.8)
)

ax.set_title('Taxa de Câmbio USD/BRL — Cotação Diária (2010–2019)', pad=14, fontsize=13, color='#e6edf3')
ax.set_xlabel('Data')
ax.set_ylabel('Reais por 1 Dólar (R$)')
ax.legend(frameon=False, fontsize=9)
ax.grid(True, axis='y')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('R$ %.2f'))

plt.tight_layout()
plt.savefig('grafico1_linha_temporal.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Gráfico 1 salvo: grafico1_linha_temporal.png')

---
## Gráfico 2 — Estatística Descritiva: Barras com Desvio Padrão

**Justificativa:** o gráfico de barras é a técnica clássica para comparar valores entre categorias discretas (anos). A sobreposição das barras de erro (±1 desvio padrão) acrescenta informação sobre a volatilidade cambial em cada período.

In [ ]:
# ─────────────────────────────────────────────
#  Gráfico 2 — Barras: média anual ± desvio padrão
# ─────────────────────────────────────────────
anual = df.groupby('Ano')['USD_BRL'].agg(['mean', 'std', 'min', 'max']).reset_index()
anual.columns = ['Ano', 'Media', 'Desvio', 'Min', 'Max']

fig, ax = plt.subplots(figsize=(12, 5))

# Cores graduadas pelo valor da média
norm   = plt.Normalize(anual['Media'].min(), anual['Media'].max())
cmap   = plt.cm.get_cmap('cool')
cores  = [cmap(norm(v)) for v in anual['Media']]

bars = ax.bar(
    anual['Ano'], anual['Media'],
    color=cores, width=0.65,
    yerr=anual['Desvio'],
    error_kw=dict(ecolor=ACCENT2, elinewidth=1.5, capsize=4, capthick=1.5),
    zorder=3
)

# Rótulos de valor nas barras
for bar, val in zip(bars, anual['Media']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.08,
        f'R${val:.2f}',
        ha='center', va='bottom', fontsize=8, color='#e6edf3'
    )

# Linha de referência (média geral)
media_geral = df['USD_BRL'].mean()
ax.axhline(media_geral, color=ACCENT3, linewidth=1.4, linestyle='--', alpha=0.7,
           label=f'Média geral R$ {media_geral:.2f}')

ax.set_title('Média Anual da Taxa USD/BRL com Desvio Padrão (2010–2019)', pad=14, fontsize=13, color='#e6edf3')
ax.set_xlabel('Ano')
ax.set_ylabel('Média da Taxa (R$)')
ax.set_xticks(anual['Ano'])
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('R$ %.2f'))
ax.legend(frameon=False, fontsize=9)
ax.grid(True, axis='y', zorder=0)

# Mini legenda da escala de cor
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02, fraction=0.025)
cbar.set_label('Taxa média (R$)', color=MUTED, fontsize=8)
cbar.ax.yaxis.set_tick_params(color=MUTED)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=MUTED, fontsize=7)

plt.tight_layout()
plt.savefig('grafico2_barras_estatistica.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Gráfico 2 salvo: grafico2_barras_estatistica.png')

---
## Gráfico 3 — Visualização Relacional/Hierárquica: Heatmap Mensal

**Justificativa:** o mapa de calor permite visualizar simultaneamente duas dimensões categóricas (ano × mês) com uma terceira variável codificada em cor (taxa média), revelando padrões sazonais e tendências de longo prazo de forma intuitiva. É uma das técnicas da unidade de visualização hierárquica/relacional.

In [ ]:
# ─────────────────────────────────────────────
#  Gráfico 3 — Heatmap: taxa média por Ano × Mês
# ─────────────────────────────────────────────
MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

pivot = df.groupby(['Ano', 'Mes'])['USD_BRL'].mean().unstack()
pivot.columns = MESES_PT  # renomeia para português

fig, ax = plt.subplots(figsize=(13, 6))

hm = sns.heatmap(
    pivot,
    ax=ax,
    cmap='RdYlGn_r',          # verde (barato) → vermelho (caro)
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    linecolor='#21262d',
    cbar_kws={'label': 'Taxa média (R$)', 'shrink': 0.8},
    annot_kws={'size': 8, 'color': '#0d1117'},
)

# Colorbar styling
cbar = hm.collections[0].colorbar
cbar.ax.yaxis.label.set_color('#c9d1d9')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#8b949e')

ax.set_title('Mapa de Calor — Taxa Média Mensal USD/BRL por Ano (2010–2019)',
             pad=16, fontsize=13, color='#e6edf3')
ax.set_xlabel('Mês', fontsize=11)
ax.set_ylabel('Ano', fontsize=11)
ax.tick_params(axis='x', labelrotation=0)
ax.tick_params(axis='y', labelrotation=0)

plt.tight_layout()
plt.savefig('grafico3_heatmap_mensal.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Gráfico 3 salvo: grafico3_heatmap_mensal.png')

---
## Conclusões

Os três gráficos são complementares e contam a mesma história sob perspectivas diferentes:

- **Gráfico 1 (Linhas):** mostra a **trajetória completa** da taxa, com destaque para o pico de 2015–2016 (crise política/econômica brasileira) e a relativa estabilidade de 2010–2013.
- **Gráfico 2 (Barras):** **quantifica e compara** a média por ano, evidenciando a desvalorização do BRL ao longo da década. As barras de erro revelam que 2015 e 2018 foram os anos mais voláteis.
- **Gráfico 3 (Heatmap):** permite identificar **padrões sazonais e anuais** simultaneamente — o calor crescente do verde para o vermelho ilustra visualmente a escalada cambial da década.

> **Dataset:** [Investing.com — USD/BRL Histórico](https://br.investing.com/currencies/usd-brl-historical-data)